# KNIME + Python: the same ML problem, two ways to build it

In this notebook we inspect one shared dataset and build a Python regression pipeline. The matching KNIME build-along is in [`../knime/README.md`](../knime/README.md).

**Learning objectives**
- Identify features, target, and metadata columns.
- Inspect missing values and categorical data.
- Use the same chronological train/test split as the KNIME tutorial.
- Build preprocessing and model training into a scikit-learn pipeline.
- Evaluate a simple baseline and a Random Forest model.

The data is synthetic and is not intended for real inventory decisions.

## 1. Import libraries and load the dataset

Run this notebook from the repository root or open it through Jupyter Lab. The path below is relative to the repository root.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('data/inventory_demand.csv')
if not DATA_PATH.exists():
    # This fallback also works when the notebook is launched from python/
    DATA_PATH = Path('../data/inventory_demand.csv')

frame = pd.read_csv(DATA_PATH)
print(f'Rows: {len(frame):,}; columns: {len(frame.columns)}')
frame.head()

## 2. Inspect the data

Before modeling, check data types, ranges, category labels, missing values, and whether the target is present. This stage is exploratory data analysis (EDA).

In [ ]:
frame.info()
frame.describe(include='all').T

In [ ]:
missing = frame.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## 3. Understand features, target, and split

The target is `demand_next_week`. The `split` column was precomputed by the dataset generator so KNIME and Python use exactly the same training and test rows. We exclude identifiers and metadata from the model inputs.

In [ ]:
TARGET = 'demand_next_week'
EXCLUDED = {'week_start', 'week_index', 'store_id', 'split', TARGET}
feature_columns = [column for column in frame.columns if column not in EXCLUDED]

train = frame.loc[frame['split'] == 'train'].copy()
test = frame.loc[frame['split'] == 'test'].copy()

print(f'Train rows: {len(train):,}')
print(f'Test rows:  {len(test):,}')
print('Features:', feature_columns)
print('Train weeks:', train['week_index'].min(), 'to', train['week_index'].max())
print('Test weeks: ', test['week_index'].min(), 'to', test['week_index'].max())

## 4. Build preprocessing + model as one pipeline

Numeric columns use median imputation. Categorical columns use most-frequent imputation and one-hot encoding. The Random Forest regressor is then fitted after preprocessing. Because the transformations are inside the pipeline, their statistics are learned only from the training rows.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = train[feature_columns].select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_features = train[feature_columns].select_dtypes(exclude=[np.number, 'bool']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

model = RandomForestRegressor(
    n_estimators=250,
    min_samples_leaf=2,
    max_features=0.85,
    random_state=42,
    n_jobs=-1,
)
pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])

## 5. Fit the model and a simple baseline

A median baseline predicts the training-set median for every test row. A useful model should beat a simple reference, not just produce a number.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train = train[feature_columns]
y_train = train[TARGET]
X_test = test[feature_columns]
y_test = test[TARGET]

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

baseline = DummyRegressor(strategy='median')
baseline.fit(np.zeros((len(y_train), 1)), y_train)
baseline_predictions = baseline.predict(np.zeros((len(y_test), 1)))

def regression_metrics(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted),
    }

results = pd.DataFrame({
    'Median baseline': regression_metrics(y_test, baseline_predictions),
    'Random Forest': regression_metrics(y_test, predictions),
}).T
results.round(3)

## 6. Inspect predictions

A metric compresses model behavior into a small number of values. A plot can reveal systematic over- or under-prediction and unusual errors.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, predictions, alpha=0.55)
limits = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
plt.plot(limits, limits, linestyle='--')
plt.xlabel('Actual demand')
plt.ylabel('Predicted demand')
plt.title('Actual vs predicted demand')
plt.tight_layout()
plt.show()

## 7. Reflect and experiment

1. What would happen if `previous_week_units` and `avg_4_week_units` were removed?
2. Which features would be unavailable at the time a real forecast is made?
3. Why should the imputer be fitted on training data only?
4. Why might the KNIME score not exactly match this Python score?
5. What would make the observed error acceptable or unacceptable for a real stock decision?

**Remember:** the Python workflow one-hot encodes categorical values. The KNIME Random Forest regression learner can handle nominal values natively, so internal representations can differ. Use the same input rows and target, but do not treat small score differences as a universal tool benchmark.